# 8 · Scenario sweeps + visualizations — distributed SeQUeNCe BB84 on FABRIC

Re-run the `qne-sequence` emulator across a distance sweep **on the FABRIC testbed**, then
cross-check it against the SeQUeNCe simulator. Both planes are raw L2 — **no TCP**: photons as
`0x7101` through the P4 switch, classical sifting/QBER as raw `0x7102` (`CLASSICAL='l2'`).

- **B.1 — distance sweep (emulation):** real `0x7101` photons; fiber loss applied by the P4 switch. QBER & sifted bits vs distance at fixed fidelity `F`.
- **B.2 — cross-check vs the SeQUeNCe simulator:** run the *same* BB84 scenarios in pure SeQUeNCe simulation (discrete-event, no wire) and overlay QBER-vs-distance — the "emulation == simulation" fidelity check. The classical channel is lossless (ANL SeQUeNCe-team guidance), so this matters more than a latency stress sweep.

Results → `qne-sequence/results/*.json`; figures inline.

> For a slice-free **local loopback** sweep, use `qne-sequence/sweep.py` + `plots.py` directly (or the validation notebooks). This notebook is FABRIC-only.

In [ ]:
import sys, json
from pathlib import Path

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'qne').is_dir())
QNE_SEQ = PROJECT_DIR / 'qne-sequence'
for p in (str(PROJECT_DIR), str(PROJECT_DIR / 'scripts'), str(QNE_SEQ)):
    if p not in sys.path:
        sys.path.insert(0, p)

import plots   # qne-sequence/plots.py — figure builders

## Prereqs & run

Run **notebook 1** (slice + data-plane IPs + BMv2 switch) and **notebook 7** once
(`setup_sequence_runtime` built `.venv-qne` on both nodes). The cells below configure the P4
switch per distance point, run the distance sweep on the real nodes, then cross-check each
point against the pure SeQUeNCe simulator (also run in `.venv-qne`, no NetSquid needed).

Both planes run as raw L2 — **no TCP**: `CLASSICAL='l2'` requires `TRANSPORT='raw'` + the
switch: `configure_switch` runs **once** and `set_channel_loss` updates the P4 loss threshold in place for every point.

For **B.2 with `REFERENCE='netsquid'`** (the trusted baseline), install NetSquid on bob once: `df.setup_sim_envs(slice_obj)` with `NETSQUID_USER`/`NETSQUID_PASS` set (netsquid.org). Use `REFERENCE='sequence'` for the switch-free SeQUeNCe sim instead.

In [ ]:
SLICE_NAME    = 'qfabric-bb84-2'
TRANSPORT     = 'raw'    # quantum photons as real 0x7101 frames through the BMv2 P4 switch
LOSS          = 'switch' # 'switch' (BMv2 P4) | 'auto'  — L2 classical needs the switch in-path
CLASSICAL     = 'l2'     # classical channel as raw 0x7102 frames (NO TCP); needs TRANSPORT='raw' + switch
ATTEN         = 0.2      # dB/km
FIDELITY      = 0.98     # polarization fidelity F -> intrinsic QBER ~ (1-F)/2
EFFICIENCY    = 0.8
DARK          = 10.0
KEY_LENGTH    = 256
SAMPLE_FRAC   = 0.2
FAB_PULSES    = 50000     # more photons -> more sifted bits -> tighter QBER (smaller error bars)
DRAIN_MS      = 500
FAB_DISTANCES = [1, 10, 25, 50]          # km (sets the loss per point)
REFERENCE     = 'netsquid'  # trusted baseline for B.2: 'netsquid' (fixed-budget BB84 +
                            # Beer-Lambert loss; needs .venv-nsq on bob) | 'sequence'

# the P4 switch is only needed for raw + switch/auto; tcp / model / none run switch-free
USE_SWITCH = (TRANSPORT == 'raw' and LOSS in ('switch', 'auto'))

import deploy_fabric as df
fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()
print(f"transport={TRANSPORT} loss={LOSS} classical={CLASSICAL} F={FIDELITY} "
      f"atten={ATTEN} dB/km -> {'using P4 switch' if USE_SWITCH else 'switch-free'}")

In [ ]:
# B.1 — distance sweep (emulation): real 0x7101 photons; fiber loss via the P4 switch
fab_rows = []
print(f"FABRIC emulation sweep at F={FIDELITY}, attenuation={ATTEN} dB/km "
      f"(intrinsic QBER ~ (1-F)/2 = {(1-FIDELITY)/2:.4f})")
if USE_SWITCH:  # start/refresh BMv2 + tables ONCE; per-point loss updated in place below
    df.configure_switch(slice_obj, int(df.loss_probability(FAB_DISTANCES[0], ATTEN) * (2**32)))
for d in FAB_DISTANCES:
    if USE_SWITCH:
        df.set_channel_loss(slice_obj, int(df.loss_probability(d, ATTEN) * (2**32)))
    a, b = df.run_sequence_bb84(
        slice_obj, transport=TRANSPORT, loss=LOSS, classical_transport=CLASSICAL,
        num_pulses=FAB_PULSES, key_length=KEY_LENGTH,
        fidelity=FIDELITY, efficiency=EFFICIENCY, dark_count_rate=DARK,
        distance_km=d, attenuation=ATTEN, sample_fraction=SAMPLE_FRAC,
        photon_mode='bulk', photon_drain_ms=DRAIN_MS)
    if a:
        a['sweep'], a['x'], a['key'], a['fidelity'] = 'distance', d, (a.get('key') is not None), FIDELITY
        fab_rows.append(a)
        print(f"  emul d={d:>3} km  F={FIDELITY}  QBER={a.get('qber')}  sifted={a.get('sifted_bits')}")

fab_out = QNE_SEQ / 'results' / 'sequence_scenarios_fabric.json'
fab_out.write_text(json.dumps(fab_rows, indent=2))
print(f"saved {len(fab_rows)} FABRIC distance points (F={FIDELITY}) -> {fab_out}")
display(plots.fig_distance(fab_rows))

In [ ]:
# B.2 — cross-check the FABRIC emulation against a TRUSTED reference simulator.
# REFERENCE='netsquid' (cf. Chan et al. 2026, the NetSquid QR paper) is a FIXED-photon-
# budget BB84 with per-photon Beer-Lambert loss + DepolarNoiseModel, so its QBER and
# yield=sifted/photons_sent are on the SAME basis as the emulation (apples-to-apples).
# 'sequence' = the SeQUeNCe sim (target-key-length; kept for comparison). The classical
# channel is lossless (ANL guidance), so the useful check is fidelity-to-sim vs distance.
import math
import matplotlib.pyplot as plt

if REFERENCE == 'netsquid':
    _run_ref = lambda d: df.run_netsquid_sim(slice_obj, distance_km=d, fidelity=FIDELITY,
                             attenuation=ATTEN, num_photons=FAB_PULSES,
                             sample_fraction=SAMPLE_FRAC, efficiency=EFFICIENCY, dark_count=DARK)
    REF_LABEL = 'NetSquid reference'
else:
    _run_ref = lambda d: df.run_sequence_sim(slice_obj, distance_km=d, fidelity=FIDELITY,
                             attenuation=ATTEN, num_photons=FAB_PULSES, sample_fraction=SAMPLE_FRAC)
    REF_LABEL = 'SeQUeNCe simulator'

def _ref_ok(r):
    return bool(r) and not (r.get('extra') or {}).get('error') \
        and r.get('qber') is not None and (r.get('sifted_bits') or 0) > 0

def _yield(r):  # sifted key bits per photon SENT
    ps = r.get('photons_sent') or 0
    return (r.get('sifted_bits') or 0) / ps if ps else 0.0

def _qber_n(row, emulation=False):
    # Bits the QBER was actually estimated over (for BOTH the error bars and the
    # agreement tolerance). The emulation discloses only a SAMPLE of its sifted key for
    # QBER and reports it as `num_sampled` (~ sifted*SAMPLE_FRAC) — NOT the full sifted
    # count. The reference sims QBER over the whole key and report `qber_sample_bits`.
    if emulation:
        n = row.get('num_sampled')
        return int(n) if n else max(int((row.get('sifted_bits') or 0) * SAMPLE_FRAC), 1)
    n = (row.get('extra') or {}).get('qber_sample_bits') or row.get('qber_sample_bits')
    return int(n) if n else max(int(row.get('sifted_bits') or 0), 1)

ref_rows = []
for d in FAB_DISTANCES:
    try:
        r = _run_ref(d)
    except Exception as exc:
        r = {'extra': {'error': repr(exc)}}
    r['x'] = d
    ref_rows.append(r)
    if _ref_ok(r):
        print(f"  {REFERENCE} d={d:>3} km  F={FIDELITY}  QBER={r.get('qber')}  "
              f"sent={r.get('photons_sent')}  sifted={r.get('sifted_bits')}  yield={_yield(r):.4f}")
    else:
        hint = "  (install NetSquid on bob: setup_sim_envs + NETSQUID_USER/PASS)" if REFERENCE == 'netsquid' else ""
        print(f"  {REFERENCE} d={d:>3} km  FAILED/SKIPPED: {(r.get('extra') or {}).get('error') or 'no result'}{hint}")

ref_out = QNE_SEQ / 'results' / f'reference_{REFERENCE}_distance.json'
ref_out.write_text(json.dumps(ref_rows, indent=2))
print(f"saved {len(ref_rows)} {REFERENCE}-reference points -> {ref_out}")

# comparable metrics: QBER (~ (1-F)/2, validates noise model) and sift-yield =
# sifted/photons_sent (falls ~ e^{-ATTEN*L}, validates loss model). A failed reference
# point is reported REF FAILED, never AGREE.
by_d = {r['x']: r for r in ref_rows}
print(f"\nEmulation vs {REF_LABEL}  (F={FIDELITY}, {ATTEN} dB/km):")
print(f"  {'dist':>6} {'emQBER':>7} {'refQBER':>7} {'emYield':>8} {'refYield':>8}  verdict")
n_agree = n_cmp = 0
for e in fab_rows:
    s = by_d.get(e['x'])
    eq = e.get('qber')
    em_yield = (e.get('sifted_bits') or 0) / FAB_PULSES if FAB_PULSES else 0.0
    if eq is None or not _ref_ok(s):
        why = 'EMUL NO QBER' if eq is None else f"REF FAILED ({(s or {}).get('extra',{}).get('error') or 'no result'})"
        print(f"  {e['x']:>4} km {'—':>7} {'—':>7} {em_yield:>8.4f} {'—':>8}  {why}")
        continue
    sq = s['qber']; ry = _yield(s)
    n_e, n_s = _qber_n(e, emulation=True), _qber_n(s, emulation=False)
    p = (eq + sq) / 2
    tol = 2 * math.sqrt(max(p * (1 - p), 1e-10) * (1 / n_e + 1 / n_s))
    ok = abs(eq - sq) <= tol
    n_cmp += 1; n_agree += int(ok)
    print(f"  {e['x']:>4} km {eq:>7.4f} {sq:>7.4f} {em_yield:>8.4f} {ry:>8.4f}  "
          f"{'AGREE' if ok else 'DIFFER'} QBER (tol {tol:.4f})")
print(f"\n{n_agree}/{n_cmp} distances agree on QBER within 2σ"
      + ("" if n_cmp == len(FAB_DISTANCES) else f"  ({len(FAB_DISTANCES) - n_cmp} point(s) had no valid comparison)"))

# --- two panels: QBER vs distance (±2σ), and sift-yield vs distance (log-y) ---
ex = [e['x'] for e in fab_rows if e.get('qber') is not None]
eq_ = [e['qber'] for e in fab_rows if e.get('qber') is not None]
ey = [(e.get('sifted_bits') or 0) / FAB_PULSES for e in fab_rows if e.get('qber') is not None]
rx = [s['x'] for s in ref_rows if _ref_ok(s)]
rq = [s['qber'] for s in ref_rows if _ref_ok(s)]
ryld = [_yield(s) for s in ref_rows if _ref_ok(s)]

def _q2sig(q, n):
    n = max(int(n or 0), 1)
    return 2 * (max(q * (1 - q), 1e-10) / n) ** 0.5
e_err = [_q2sig(e['qber'], _qber_n(e, emulation=True))
         for e in fab_rows if e.get('qber') is not None]
r_err = [_q2sig(s['qber'], _qber_n(s, emulation=False))
         for s in ref_rows if _ref_ok(s)]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].errorbar(ex, eq_, yerr=e_err, fmt='o-', capsize=3, label='FABRIC emulation')
if rx: ax[0].errorbar(rx, rq, yerr=r_err, fmt='s--', capsize=3, label=REF_LABEL)
ax[0].axhline((1 - FIDELITY) / 2, color='gray', ls=':', label=f'(1-F)/2 = {(1 - FIDELITY) / 2:.4f}')
ax[0].set(xlabel='distance (km)', ylabel='QBER', title=f'QBER vs distance  (F={FIDELITY}, ±2σ)')
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].semilogy(ex, ey, 'o-', label='FABRIC emulation (sifted / FAB_PULSES)')
if rx: ax[1].semilogy(rx, ryld, 's--', label=f'{REF_LABEL} (sifted / photons_sent)')
ax[1].set(xlabel='distance (km)', ylabel='sift yield (log)',
          title=f'Sift-yield vs distance  ({ATTEN} dB/km — both ~ e^(-αL))')
ax[1].legend(); ax[1].grid(alpha=0.3, which='both')
plt.suptitle(f'Emulation vs {REF_LABEL}  (F={FIDELITY}, {ATTEN} dB/km)')
plt.tight_layout(); plt.show()

**Interpreting B** — two comparable metrics vs the reference simulator:
- **QBER** should sit at ≈ `(1-F)/2` on both and agree within the 2σ error bars — validates the
  **noise model**.
- **Sift-yield** (sifted bits per photon sent) falls ≈ `e^(-αL)` on both — validates the **loss
  model**. With the NetSquid reference (a fixed-photon-budget BB84, same as the emulation) the two
  curves should nearly overlap; residual offset reflects detector-model differences, so read the
  slope. (SeQUeNCe's target-key-length model matches only in slope, not absolute yield.)

The classical `0x7102` channel is lossless, so it perturbs neither metric — which is why we validate
against a simulator rather than stress it with netem.

Cleanup: `df.cleanup(fablib, SLICE_NAME)` deletes the slice.